## **Transcription vs. Dictionary Comparison**
Once the OCR model generates a transcription, we need to evaluate its word-level accuracy. The basic approach is to extract all unique words from the predicted text and cross-reference them against a reference dictionary. This allows us to identify misspellings, flag out-of-vocabulary terms, and capture valid alternative spellings.

In [ ]:
import json
import os
import re
from collections import Counter
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from rapidfuzz import fuzz, process

load_dotenv()
PROJECT_ROOT = os.environ["PROJECT_ROOT"]

In [ ]:
transcription_dir = Path(PROJECT_ROOT, "data/processed/transcription/ocr_kept_20260515_104644")
occ_dictionary_path = Path(PROJECT_ROOT, "data/raw/DOM_lemma_variants.json")

In [ ]:
# Read every .txt file in the transcription folder (top-level only — does NOT
# descend into the per-line subdirectories). Each file is a full page; we
# concatenate them into one big text blob for word-frequency analysis.
txt_files = sorted(transcription_dir.glob("*.txt"))
text_parts = []
for txt_file in txt_files:
    with open(txt_file, "r", encoding="utf-8") as fh:
        text_parts.append(fh.read().lower())
text = "\n".join(text_parts)

words = re.findall(r"\b\w+\b", text)
freq_dict_trans = dict(Counter(words))

with open(occ_dictionary_path, "r", encoding="utf-8") as f:
    occ_dictionary = json.load(f)
occ_dictionary = {
    k.lower(): (v if v is None else [item.lower() for item in v])
    for k, v in occ_dictionary.items()
}

print(f"Loaded {len(txt_files)} transcription file(s) from {transcription_dir.name}/")
print(f"OCR vocab: {len(freq_dict_trans)} unique tokens, {sum(freq_dict_trans.values())} total")
print(f"Dictionary: {len(occ_dictionary)} headwords")

In [24]:
len(freq_dict_trans)

384

In [25]:
type(freq_dict_trans)

dict

## How the fuzzy matching works

For every OCR token that isn't an exact match in the dictionary, we look for the *closest* dictionary entry using `rapidfuzz.fuzz.ratio`. The score is based on **Indel distance** — the minimum number of single-character insertions and deletions to transform one string into the other (a substitution counts as 1 deletion + 1 insertion = 2 edits).

The score is normalized to 0–100:

$$\text{score} = 100 \times \left(1 - \frac{\text{indel\_distance}}{|s_1| + |s_2|}\right)$$

Examples:
- `"necessaria"` vs `"necesaria"` → 1 missing character → score ≈ 94.7
- `"delu"` vs `"del"` → 1 extra character → score ≈ 85.7
- `"lequal"` vs `"eyqual"` → 4 edits → score ≈ 66.7

`process.extract(query, choices, scorer=fuzz.ratio, limit=1)` runs the query against every entry in `choices` and returns the best match.

### Why short words need a stricter rule

A single-character change in a short word is a much larger fraction of the word than the same change in a long one, so short OCR tokens pass the threshold too easily. We compensate with a **length-aware threshold**:

| OCR word length | Fuzzy behaviour |
|---|---|
| ≤ 4 chars | skipped — require exact match only |
| 5 chars | base threshold **+ 10** (stricter) |
| ≥ 6 chars | base threshold |

We also drop OCR tokens shorter than `min_word_length` (default 2) before evaluating, to filter out single-letter OCR noise like `"t"`, `"q"`, `"d"`. **Important:** this is a hard drop from the entire evaluation — set it conservatively (2 is usually right). It is *not* the same as the fuzzy cutoff above: a 3-letter OCR word is still evaluated, it just has to *exactly* match the dictionary to count.

In [ ]:
def normalize_old_occitan(word: str) -> str:
    """Lowercase and keep only alphabetic characters (strip digits, punctuation, spaces)."""
    return "".join(c.lower() for c in word if c.isalpha())


def prepare_occitan_dict(custom_dict: dict):
    """
    Expand headwords + their variants into a flat set of valid forms, plus a
    reverse map from any valid form to its canonical headword.

    If a variant appears under multiple headwords, the first one wins
    (JSON insertion order). Headwords always take precedence over variants.
    """
    valid_forms = set()
    form_to_headword = {}

    for head, variants in custom_dict.items():
        h_norm = normalize_old_occitan(head)
        if h_norm:
            valid_forms.add(h_norm)
            form_to_headword[h_norm] = head

        if isinstance(variants, list):
            for var in variants:
                v_norm = normalize_old_occitan(var)
                if v_norm:
                    valid_forms.add(v_norm)
                    form_to_headword.setdefault(v_norm, head)

    return valid_forms, form_to_headword


def length_aware_threshold(word: str, base_threshold: float):
    """
    Return the fuzzy threshold to use for `word`, or None to skip fuzzy
    matching entirely. See the markdown cell above for rationale.

      length <= 4  -> None (require exact match)
      length == 5  -> base_threshold + 10 (stricter)
      length >= 6  -> base_threshold
    """
    n = len(word)
    if n <= 4:
        return None
    if n <= 5:
        return min(base_threshold + 10, 100)
    return base_threshold


def evaluate_ocr_occitan(
    ocr_counter,
    custom_dict,
    fuzzy_threshold: float = 82,
    min_word_length: int = 2,
):
    """
    Evaluate an OCR transcription against an Occitan dictionary.

    Returns a dict containing:
      - exact and combined (exact + fuzzy) word/token accuracy
      - OOV (out-of-vocabulary) counts after fuzzy matching
      - DataFrame of fuzzy match candidates
      - all best-match fuzzy scores (for histogram / threshold tuning)
      - frequency-sorted list of words that matched nothing
    """
    valid_forms, form_to_headword = prepare_occitan_dict(custom_dict)
    valid_list = sorted(valid_forms)

    # Normalize OCR tokens, drop anything shorter than min_word_length
    # (filters out single-letter OCR noise like "t", "q", "d").
    ocr_norm_map = {}  # {normalized_word: [original_word, count]}
    dropped_short_tokens = 0
    for word, count in ocr_counter.items():
        w_norm = normalize_old_occitan(word)
        if not w_norm:
            continue
        if len(w_norm) < min_word_length:
            dropped_short_tokens += count
            continue
        if w_norm not in ocr_norm_map:
            ocr_norm_map[w_norm] = [word, 0]
        ocr_norm_map[w_norm][1] += count

    ocr_norm_words = set(ocr_norm_map.keys())

    # 1. Exact matches
    exact_matches = ocr_norm_words & valid_forms
    unknown_words = ocr_norm_words - valid_forms

    total_tokens = sum(c for _, c in ocr_norm_map.values())
    exact_tokens = sum(ocr_norm_map[w][1] for w in exact_matches)

    word_acc_exact = len(exact_matches) / len(ocr_norm_words) if ocr_norm_words else 0
    token_acc_exact = exact_tokens / total_tokens if total_tokens else 0

    # 2. Fuzzy matching for unknowns, with a length-aware threshold
    fuzzy_candidates = []
    fuzzy_scores = []          # every attempted best score, for tuning
    fuzzy_matched_norm = set()  # unknowns that passed their threshold

    for u_norm in unknown_words:
        threshold = length_aware_threshold(u_norm, fuzzy_threshold)
        if threshold is None:
            continue  # too short to fuzzy-match safely

        orig, freq = ocr_norm_map[u_norm]
        matches = process.extract(u_norm, valid_list, limit=1, scorer=fuzz.ratio)
        if not matches:
            continue
        best_form, best_score, _ = matches[0]
        fuzzy_scores.append(best_score)

        if best_score >= threshold:
            fuzzy_matched_norm.add(u_norm)
            headword = form_to_headword.get(best_form, best_form)
            fuzzy_candidates.append({
                "ocr_original": orig,
                "ocr_normalized": u_norm,
                "frequency": freq,
                "matched_form": best_form,
                "suggested_headword": headword,
                "fuzzy_score": best_score,
                "word_length": len(u_norm),
                "threshold_used": threshold,
            })

    fuzzy_df = pd.DataFrame(fuzzy_candidates)
    if not fuzzy_df.empty:
        fuzzy_df = fuzzy_df.sort_values("frequency", ascending=False).reset_index(drop=True)

    # 3. Combined (exact + fuzzy) accuracy
    fuzzy_tokens = sum(ocr_norm_map[w][1] for w in fuzzy_matched_norm)
    combined_word_acc = (
        (len(exact_matches) + len(fuzzy_matched_norm)) / len(ocr_norm_words)
        if ocr_norm_words else 0
    )
    combined_token_acc = (
        (exact_tokens + fuzzy_tokens) / total_tokens if total_tokens else 0
    )

    # 4. Truly unmatched (OOV after fuzzy), sorted by frequency
    oov_norm = unknown_words - fuzzy_matched_norm
    top_unmatched = sorted(
        (
            {
                "ocr_original": ocr_norm_map[w][0],
                "ocr_normalized": w,
                "frequency": ocr_norm_map[w][1],
            }
            for w in oov_norm
        ),
        key=lambda r: r["frequency"],
        reverse=True,
    )
    top_unmatched_df = pd.DataFrame(top_unmatched)

    return {
        # Exact-match metrics
        "word_accuracy_exact": word_acc_exact,
        "token_accuracy_exact": token_acc_exact,
        # Combined (exact + fuzzy) metrics
        "word_accuracy_with_fuzzy": combined_word_acc,
        "token_accuracy_with_fuzzy": combined_token_acc,
        # Counts
        "total_unique_words": len(ocr_norm_words),
        "total_tokens": total_tokens,
        "known_words_exact": len(exact_matches),
        "fuzzy_match_words": len(fuzzy_matched_norm),
        "oov_words": len(oov_norm),
        "oov_tokens": sum(ocr_norm_map[w][1] for w in oov_norm),
        "dropped_short_tokens": dropped_short_tokens,
        # Per-row data
        "fuzzy_candidates": fuzzy_df,
        "fuzzy_scores": pd.Series(fuzzy_scores, name="fuzzy_score"),
        "top_unmatched": top_unmatched_df,
    }

In [ ]:
results = evaluate_ocr_occitan(
    freq_dict_trans,
    occ_dictionary,
    fuzzy_threshold=82,
    min_word_length=2,  # drop only single-char tokens (OCR noise)
)

In [27]:
results

{'type_accuracy': 0.3394255874673629,
 'token_accuracy': 0.5151515151515151,
 'known_types': 130,
 'unknown_types': 253,
 'total_unique_words': 383,
 'fuzzy_candidates':     ocr_original ocr_normalized  frequency matched_form suggested_headword  \
 46          delu           delu         13          del                del   
 162       aquela         aquela          7        aquel              aquel   
 107         dela           dela          6        delai              delai   
 7           quar           quar          5        quiar              quiar   
 8         lequal         lequal          5       eyqual             aiquel   
 ..           ...            ...        ...          ...                ...   
 72      treyssec       treyssec          1    entressec          entressec   
 73         delas          delas          1         dels                del   
 74    necessaria     necessaria          1    necesaria          necesaria   
 75         perla          perla         

In [ ]:
# Concise summary of all metrics
print("=" * 60)
print("OCR vs. Dictionary Evaluation")
print("=" * 60)
print(f"Total unique words (after filtering):  {results['total_unique_words']:>6}")
print(f"Total tokens:                          {results['total_tokens']:>6}")
print(f"Dropped (too-short OCR noise):         {results['dropped_short_tokens']:>6}")
print()
print(f"Exact matches      (words):            {results['known_words_exact']:>6}")
print(f"Fuzzy matches      (words):            {results['fuzzy_match_words']:>6}")
print(f"Out-of-vocabulary  (words):            {results['oov_words']:>6}")
print(f"Out-of-vocabulary  (tokens):           {results['oov_tokens']:>6}")
print()
print(f"Word accuracy  (exact only):    {results['word_accuracy_exact']*100:>5.1f}%")
print(f"Word accuracy  (exact + fuzzy): {results['word_accuracy_with_fuzzy']*100:>5.1f}%")
print(f"Token accuracy (exact only):    {results['token_accuracy_exact']*100:>5.1f}%")
print(f"Token accuracy (exact + fuzzy): {results['token_accuracy_with_fuzzy']*100:>5.1f}%")

In [ ]:
# Distribution of best fuzzy scores. Use this to pick / tune `fuzzy_threshold`:
# look for a natural gap or elbow in the histogram, and set the threshold there.
import matplotlib.pyplot as plt

scores = results["fuzzy_scores"]
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(scores, bins=25, edgecolor="black")
ax.axvline(82, color="red", linestyle="--", label="base threshold = 82")
ax.set_xlabel("fuzzy score (rapidfuzz.fuzz.ratio)")
ax.set_ylabel("number of unknown OCR words")
ax.set_title(f"Best fuzzy score per unknown OCR word (n = {len(scores)})")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Elbow-method plots to help choose `fuzzy_threshold`.
#
# Left panel: best fuzzy score per unknown word, sorted descending. The "elbow"
# is where the curve has its sharpest bend — scores above it are likely real
# matches; scores below it are likely noise. Set the threshold at the elbow.
#
# Right panel: how many candidates would be accepted at each candidate
# threshold. Use this to see the practical cost of raising / lowering it
# (e.g. "moving from 82 -> 88 drops me from 98 to 60 matches").
import numpy as np

scores_arr = np.array(results["fuzzy_scores"])
scores_sorted = np.sort(scores_arr)[::-1]
ranks = np.arange(1, len(scores_sorted) + 1)

thresholds = np.arange(60, 101, 1)
counts = [(scores_arr >= t).sum() for t in thresholds]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(ranks, scores_sorted, marker=".", linestyle="-")
axes[0].axhline(82, color="red", linestyle="--", label="threshold = 82")
axes[0].set_xlabel("rank (1 = highest-scoring unknown word)")
axes[0].set_ylabel("fuzzy score")
axes[0].set_title("Sorted scores — look for the elbow")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(thresholds, counts, marker=".")
axes[1].axvline(82, color="red", linestyle="--", label="threshold = 82")
axes[1].set_xlabel("candidate threshold")
axes[1].set_ylabel("number of unknowns accepted")
axes[1].set_title("Coverage vs. threshold")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Top-20 words the OCR produced that neither exact-matched nor fuzzy-matched.
# These are the highest-leverage errors to inspect (recurring OCR confusions,
# real Occitan words missing from the dictionary, etc.).
results["top_unmatched"].head(20)